## Section 1: The Configuration & Environment Setup
*We centralize all hyperparameters here. This makes Ablation Studies easy. We also initialize our GPU and import required libraries.*

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# --- HYPERPARAMETERS & CONFIG ---
CONFIG = {
    "n_points": 1024,
    "batch_size": 64,     # Note: PointNet++ is heavier, you might need to drop to 16 if Colab runs out of GPU memory
    "num_classes": 40,
    "use_augmentation": True,
    "epochs": 150,
    "learning_rate": 0.001,
    "weight_decay": 0.0001
}

# Setup Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Training on: {device}")

# Optional: Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


Training on: cuda


## Section 2: Data Extraction

In [3]:
from google.colab import drive
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
zip_path = '/content/drive/MyDrive/Colab Notebooks/AI535_Final_Project/ModelNet40.zip'
# zip_path = '/content/drive/MyDrive/ModelNet40.zip'
local_extract_path = '/content/data'
root_dir = '/content/data/ModelNet40'

# 3. Extract if not already extracted in this session
if not os.path.exists(root_dir):
    print("Extracting ModelNet40 to local Colab storage. This may take a minute...")
    !unzip -q "{zip_path}" -d {local_extract_path}
    print("Extraction complete!")
else:
    print("Data is already extracted and ready to go.")

Mounted at /content/drive
Extracting ModelNet40 to local Colab storage. This may take a minute...
Extraction complete!


## Section 3: Dataset, Loader, & Weights

In [4]:
# Dynamically map the 40 classes
classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

def load_off(file_path, n_points=1024):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    if lines[0].strip() == 'OFF':
        n_verts = int(lines[1].strip().split()[0])
        verts_start = 2
    else:
        n_verts = int(lines[0].strip()[3:].split()[0])
        verts_start = 1

    verts = np.array([ [float(x) for x in l.strip().split()] for l in lines[verts_start:verts_start + n_verts] ])

    if len(verts) >= n_points:
        indices = np.random.choice(len(verts), n_points, replace=False)
    else:
        indices = np.random.choice(len(verts), n_points, replace=True)
    verts = verts[indices]

    # Normalization
    verts = verts - np.mean(verts, axis=0)
    dist = np.max(np.sqrt(np.sum(verts**2, axis=1)))
    verts = verts / dist

    return torch.tensor(verts, dtype=torch.float32)

class FastModelNetDataset(Dataset):
    def __init__(self, root, split='train', n_points=1024):
        self.n_points = n_points
        self.split = split
        self.files = []
        for cls_name, idx in class_to_idx.items():
            cls_folder = os.path.join(root, cls_name, split)
            if os.path.exists(cls_folder):
                for f in os.listdir(cls_folder):
                    if f.endswith('.off'):
                        self.files.append({'path': os.path.join(cls_folder, f), 'label': idx})

        print(f"Parsing and caching {len(self.files)} {split} files into RAM...")
        self.data = []
        self.labels = []
        for item in self.files:
            points = load_off(item['path'], self.n_points)
            self.data.append(points.transpose(0, 1))
            self.labels.append(item['label'])
        print(f"Finished caching {split} split!")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        # Clone the data so we don't accidentally overwrite our clean RAM cache!
        points = self.data[idx].clone()
        label = self.labels[idx]

        # --- DYNAMIC DATA AUGMENTATION (Training Set Only) ---
        if self.split == 'train' and CONFIG.get("use_augmentation", False):
            # 1. Random Rotation (around the Up/Y-axis)
            theta = random.uniform(0, 2 * np.pi)
            rot_matrix = torch.tensor([
                [np.cos(theta), 0, np.sin(theta)],
                [0, 1, 0],
                [-np.sin(theta), 0, np.cos(theta)]
            ], dtype=torch.float32)

            # Multiply (3,3) matrix by our (3, 1024) point cloud
            points = torch.matmul(rot_matrix, points)

            # 2. Random Gaussian Jitter (Noise)
            noise = torch.randn_like(points) * 0.02
            points = points + noise

        return points, label

# Initialize Datasets
trainset = FastModelNetDataset(root_dir, split='train', n_points=CONFIG["n_points"])
testset = FastModelNetDataset(root_dir, split='test', n_points=CONFIG["n_points"])

trainloader = DataLoader(trainset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2)

# --- CLASS IMBALANCE CALCULATION ---
print("\nCalculating class weights for Weighted Cross-Entropy...")
class_counts = np.zeros(CONFIG["num_classes"])
for label in trainset.labels:
    class_counts[label] += 1

# w_c = N_total / (C * N_c)
total_samples = len(trainset.labels)
class_weights = total_samples / (CONFIG["num_classes"] * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights calculated successfully!")

Parsing and caching 9843 train files into RAM...
Finished caching train split!
Parsing and caching 2468 test files into RAM...
Finished caching test split!

Calculating class weights for Weighted Cross-Entropy...
Class weights calculated successfully!


## Section 4: PointNet++ Geometric Math
*To overcome the "local blindness" of Global Max Pooling, PointNet++ requires custom functions to measure distances, sample the furthest points (FPS), and group neighbors into spheres.*

In [5]:
def square_distance(src, dst):
    B, N, _ = src.shape
    _, M, _ = dst.shape
    dist = -2 * torch.matmul(src, dst.transpose(1, 2))
    dist += torch.sum(src ** 2, -1).view(B, N, 1)
    dist += torch.sum(dst ** 2, -1).view(B, 1, M)
    return dist

def index_points(points, idx):
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

def farthest_point_sample(xyz, npoint):
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids

def query_ball_point(radius, nsample, xyz, new_xyz):
    device = xyz.device
    B, N, C = xyz.shape
    S = new_xyz.shape[1]
    group_idx = torch.arange(N, dtype=torch.long).to(device).view(1, 1, N).repeat([B, S, 1])
    sqrdists = square_distance(new_xyz, xyz)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat([1, 1, nsample])
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx

## Section 5: Model Architecture (Set Abstraction)
*Here we define the hierarchical structure. We use Set Abstraction (SA) layers to group points into local neighborhoods, extracting features before finally applying global pooling on a much smaller, richer set of features.*

In [6]:
class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint = npoint
        self.radius = radius
        self.nsample = nsample
        self.group_all = group_all
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz, points):
        xyz = xyz.permute(0, 2, 1)
        if points is not None:
            points = points.permute(0, 2, 1)

        if self.group_all:
            new_xyz, new_points = xyz, points
            grouped_xyz = xyz.view(xyz.shape[0], 1, xyz.shape[1], 3)
            if points is not None:
                new_points = torch.cat([grouped_xyz, points.view(points.shape[0], 1, points.shape[1], -1)], dim=-1)
            else:
                new_points = grouped_xyz
        else:
            new_xyz_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, new_xyz_idx)
            idx = query_ball_point(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, idx)
            grouped_xyz -= new_xyz.view(new_xyz.shape[0], new_xyz.shape[1], 1, 3)

            if points is not None:
                grouped_points = index_points(points, idx)
                new_points = torch.cat([grouped_xyz, grouped_points], dim=-1)
            else:
                new_points = grouped_xyz

        new_points = new_points.permute(0, 3, 2, 1)
        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            new_points = F.relu(bn(conv(new_points)))

        new_points = torch.max(new_points, 2)[0]
        return new_xyz.permute(0, 2, 1), new_points

class PointNet2ClsSSG(nn.Module):
    def __init__(self, num_classes=40):
        super(PointNet2ClsSSG, self).__init__()
        self.sa1 = PointNetSetAbstraction(512, 0.2, 32, 3, [64, 64, 128], False)
        self.sa2 = PointNetSetAbstraction(128, 0.4, 64, 128 + 3, [128, 128, 256], False)
        self.sa3 = PointNetSetAbstraction(None, None, None, 256 + 3, [256, 512, 1024], True)

        self.fc1 = nn.Linear(1024, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.drop2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, xyz):
        B = xyz.shape[0]
        l1_xyz, l1_points = self.sa1(xyz, None)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        l3_xyz, l3_points = self.sa3(l2_xyz, l2_points)

        x = l3_points.view(B, 1024)
        x = self.drop1(F.relu(self.bn1(self.fc1(x))))
        x = self.drop2(F.relu(self.bn2(self.fc2(x))))
        x = self.fc3(x)
        return x # Return logits for CrossEntropyLoss

## Section 6: Training & Evaluation Loop
*We use AdamW with weight decay. Notice the training loop prints the overall accuracy to help us track the "Generalization Tax."*

In [ ]:
# 1. Import and Initialize wandb
import wandb
wandb.login() # This will prompt you for your API key if not already logged in
wandb.init(
    project="AI535_Final_Project",
    name="PointNet++_SSG",
    config=CONFIG
)

# 2. Instantiate Model, Optimizer, and Loss (Fixed variable names)
model = PointNet2ClsSSG(num_classes=CONFIG["num_classes"]).to(device)
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])

# 3. Use the calculated weights to fight Class Imbalance!
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

best_test_acc = 0.0

print("Starting PointNet++ Training Loop...")
for epoch in range(CONFIG["epochs"]):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    # Iterate over trainloader (Fixed name)
    for points, targets in trainloader:
        points, targets = points.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(points)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

    train_acc = 100 * correct / total
    avg_train_loss = train_loss / len(trainloader)

    # Validation Loop
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        # Iterate over testloader (Fixed name)
        for points, targets in testloader:
            points, targets = points.to(device), targets.to(device)
            outputs = model(points)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_total += targets.size(0)
            test_correct += (predicted == targets).sum().item()

    test_acc = 100 * test_correct / test_total
    avg_test_loss = test_loss / len(testloader)

    # Save the best model
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'best_pointnet2.pth')

    print(f"Epoch [{epoch+1}/{CONFIG['epochs']}] - Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% (Best: {best_test_acc:.2f}%)")

    # 4. Log metrics to Weights & Biases
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_acc": train_acc,
        "test_loss": avg_test_loss,
        "test_acc": test_acc,
        "best_test_acc": best_test_acc
    })

# Finish the wandb run
wandb.finish()
print("Training Complete. Model saved as 'best_pointnet2.pth'.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yushe (yushe-oregon-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting PointNet++ Training Loop...
Epoch [1/150] - Train Loss: 2.9424, Train Acc: 20.09% | Test Acc: 28.69% (Best: 28.69%)
Epoch [2/150] - Train Loss: 2.4106, Train Acc: 33.10% | Test Acc: 29.21% (Best: 29.21%)
Epoch [3/150] - Train Loss: 2.2090, Train Acc: 38.70% | Test Acc: 28.93% (Best: 29.21%)
Epoch [4/150] - Train Loss: 2.0501, Train Acc: 41.90% | Test Acc: 28.73% (Best: 29.21%)
Epoch [5/150] - Train Loss: 1.9501, Train Acc: 45.17% | Test Acc: 33.06% (Best: 33.06%)
Epoch [6/150] - Train Loss: 1.8573, Train Acc: 47.29% | Test Acc: 33.71% (Best: 33.71%)
Epoch [7/150] - Train Loss: 1.7592, Train Acc: 49.60% | Test Acc: 30.43% (Best: 33.71%)
Epoch [8/150] - Train Loss: 1.6233, Train Acc: 53.77% | Test Acc: 32.21% (Best: 33.71%)
Epoch [9/150] - Train Loss: 1.5418, Train Acc: 55.51% | Test Acc: 34.56% (Best: 34.56%)
Epoch [10/150] - Train Loss: 1.4173, Train Acc: 58.95% | Test Acc: 34.24% (Best: 34.56%)
Epoch [11/150] - Train Loss: 1.3684, Train Acc: 59.71% | Test Acc: 38.49% (Best: 3

## Section 7: Quantitative Evaluation (OA & mAcc)
*Loads the best saved weights and calculates Overall Accuracy and Mean Per-Class Accuracy to prove the effectiveness of the Weighted Loss.*

In [1]:
import torch

# 1. Surgically remove W&B hooks (prevents memory leaks if you re-run cells)
for module in model.modules():
    module._forward_hooks.clear()
    module._backward_hooks.clear()
    module._forward_pre_hooks.clear()

# 2. Load the BEST PointNet++ weights from our checkpoint
print("Loading Best PointNet++ Checkpoint ('best_pointnet2.pth')...")
model.load_state_dict(torch.load('best_pointnet2.pth'))
model.eval()

total, correct = 0, 0
class_correct = torch.zeros(CONFIG["num_classes"]).to(device)
class_total = torch.zeros(CONFIG["num_classes"]).to(device)

print("Evaluating Best Checkpoint on Test Set...")
with torch.no_grad():
    for points, labels in testloader:
        points, labels = points.to(device), labels.to(device)
        outputs = model(points)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        c = (predicted == labels)
        for i in range(labels.size(0)):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1

oa = 100.0 * correct / total
per_class_acc = class_correct / torch.clamp(class_total, min=1)
macc = 100.0 * torch.mean(per_class_acc).item()

print("\n" + "=" * 40)
print(f"Final Overall Accuracy (OA):       {oa:.2f}%")
print(f"Final Mean Per-Class Acc (mAcc):   {macc:.2f}%")
print("=" * 40)

print("\nLowest performing classes (The 'Hard' Classes):")
acc_dict = {classes[i]: (per_class_acc[i].item() * 100) for i in range(CONFIG["num_classes"])}
sorted_acc = sorted(acc_dict.items(), key=lambda x: x[1])
for cls_name, acc in sorted_acc[:5]:
    print(f"{cls_name:>15}: {acc:.2f}%")

NameError: name 'model' is not defined

## Section 8: Qualitative Results (10-Class Visualization Grid)
*Generates a 3D scatter plot grid of 10 distinct classes to visually inspect the model's performance.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Gathering predictions for visualization...")
model.eval()
all_candidates = []

with torch.no_grad():
    for points, labels in testloader:
        points_gpu = points.to(device)
        outputs = model(points_gpu)
        _, preds = torch.max(outputs, 1)

        for i in range(points.size(0)):
            all_candidates.append({
                'pc': points[i].cpu().numpy(),   # Shape is [3, N]
                'pred': preds[i].item(),
                'true': labels[i].item()
            })

# Sort so we prioritize showing a mix, but we'll extract 10 distinct predictions
selected_samples = []
seen_preds = set()

for cand in all_candidates:
    if cand['pred'] not in seen_preds:
        seen_preds.add(cand['pred'])
        selected_samples.append(cand)
    if len(selected_samples) == 10:
        break

print("Plotting the 3D Grid...")
fig = plt.figure(figsize=(22, 9))
fig.suptitle("PointNet++ Predictions (10 Distinct Classes)", fontsize=20, y=0.98)

for i, sample in enumerate(selected_samples):
    ax = fig.add_subplot(2, 5, i + 1, projection='3d')
    pc = sample['pc']

    # PyTorch to Matplotlib shape correction: [3, N] -> [N, 3]
    if pc.shape[0] == 3:
        pc = pc.transpose(1, 0)

    # Normalize point cloud so the camera view is perfectly centered
    pc = pc - np.mean(pc, axis=0)
    scale = np.max(np.linalg.norm(pc, axis=1))
    if scale > 0:
        pc = pc / scale

    x, y, z = pc[:, 0], pc[:, 1], pc[:, 2]

    # Z-axis color map for depth perception
    ax.scatter(x, y, z, s=12, c=z, cmap='viridis', alpha=0.9)

    pred_name = classes[sample['pred']]
    true_name = classes[sample['true']]
    is_correct = sample['pred'] == sample['true']

    title_color = 'green' if is_correct else 'red'
    ax.set_title(
        f"Pred: {pred_name}\n(True: {true_name})",
        color=title_color,
        fontsize=14,
        fontweight='bold',
        pad=10
    )

    # Lock camera angle and remove messy grid lines
    ax.view_init(elev=20, azim=45)
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([-1, 1])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.grid(False)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("pointnet2_predictions.png", bbox_inches='tight', dpi=300)
plt.show()
print("Visualization saved successfully as 'pointnet2_predictions.png'.")